In [1]:
import pandas as pd

# Load the Excel file
file_path = r"C:\Users\Administrator\Desktop\TAO_data\spaces_vote_link_v4.xlsx"
df = pd.read_excel(file_path, sheet_name='spaces_DATA')

# Step 1: Clean 'UCID' column
df = df[df['UCID'].notna()]  # Remove missing (NaN) UCID
df = df[~df['UCID'].astype(str).str.lower().isin(['', 'n/a', 'not sure'])]

# Step 2: Clean 'Name_Defillama' column
df = df[df['Name_Defillama'].notna()]  # Remove NaN
df = df[~df['Name_Defillama'].astype(str).str.lower().isin(['', 'n/a'])]

# Step 3: Keep only specific columns
final_df = df[['tokenid', 'UCID', 'Name_Defillama', 'Symbol_Defillama']]

# Output the final result to a new Excel file (optional)
output_path = r"C:\Users\Administrator\Desktop\TAO_data\spaces_vote_link_cleaned.xlsx"
# final_df.to_excel(output_path, index=False)

print("Processing complete. Cleaned file saved.")


Processing complete. Cleaned file saved.


In [2]:
final_df

,tokenid,UCID,Name_Defillama,Symbol_Defillama
1,stgdao.eth,18934,Stargate Finance,STG
2,arbitrumfoundation.eth,11841,Arbitrum DAO,ARB
3,aave.eth,7278,AAVE,AAVE
4,linea-build.eth,27657,Linea,NaN
5,opcollective.eth,11840,Optimism,OP
...,...,...,...,...
1168,ampleforthorg.eth,9421,Ampleforth Governance,FORTH
1170,realdefiplaza.eth,14636,DefiPlaza,DFP2
1171,leaguedao.eth,14131,LeagueDAO,LEAG
1173,convergencefinance.eth,8716,Convergence,CONV


In [3]:
import os

# Step 4: Load all filenames from the final_tvl folder
folder_path = r"C:\Users\Administrator\Desktop\TAO_data\final_tvl"
file_names = os.listdir(folder_path)
file_names_lower = [f.lower() for f in file_names]  # normalize case

# Step 5: Initialize lists for match status and matching file(s)
match_status = []
matching_files = []

# Step 6: Loop through each row to find matches
for _, row in final_df.iterrows():
    name = str(row['Name_Defillama']).strip().lower()
    symbol = str(row['Symbol_Defillama']).strip().lower()

    # Check for exact match by name
    exact_matches = [f for f in file_names if f.lower() == name]

    if exact_matches:
        match_status.append("Match")
        matching_files.append(exact_matches[0])
    else:
        # Build candidates list using name
        candidates = [f for f in file_names if name in f.lower()]

        # Add symbol-based candidates only if symbol is non-empty and not 'nan'
        if symbol and symbol != 'nan':
            symbol_candidates = [f for f in file_names if symbol in f.lower()]
            candidates = list(set(candidates + symbol_candidates))  # merge and deduplicate

        if candidates:
            match_status.append("Candidate")
            matching_files.append("; ".join(candidates))
        else:
            match_status.append("")
            matching_files.append("")

# Step 7: Add match results to the DataFrame
final_df['Match_Status'] = match_status
final_df['Matching_File'] = matching_files

# Step 8: Save the final DataFrame with matching info
matched_output_path = r"C:\Users\Administrator\Desktop\TAO_data\spaces_vote_link_matched.xlsx"
final_df.to_excel(matched_output_path, index=False)

print("Matching complete. Final file saved:", matched_output_path)


Matching complete. Final file saved: C:\Users\Administrator\Desktop\TAO_data\spaces_vote_link_matched.xlsx


C:\Users\Administrator\AppData\Local\Temp\ipykernel_30108\1505616990.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['Match_Status'] = match_status
C:\Users\Administrator\AppData\Local\Temp\ipykernel_30108\1505616990.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['Matching_File'] = matching_files


In [4]:
import pandas as pd
import os

# Load the UCID-matching file info
matched_df = pd.read_excel(r"C:\Users\Administrator\Desktop\TAO_data\TVL_matched.xlsx")

# Directory where the protocol TVL files are stored
data_dir = r"C:\Users\Administrator\Desktop\TAO_data\final_tvl"

# To store final UCID-wise results
final_list = []

# Group by UCID
for ucid, group in matched_df.groupby('UCID'):
    # Collect all matching files for this UCID
    file_candidates = group['Matching_File'].dropna().tolist()
    
    all_files = set()
    for f in file_candidates:
        if isinstance(f, str):
            files = [x.strip() for x in f.split(";") if x.strip()]
            all_files.update(files)

    dfs = []

    for file in all_files:
        file_path = os.path.join(data_dir, file)
        if not os.path.exists(file_path):
            continue  # skip missing files

        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"Skipping {file} due to read error: {e}")
            continue

        # Validate file format: should contain exactly two columns
        if df.shape[1] != 2 or 'date' not in df.columns:
            print(f"Skipping {file}: incorrect format")
            continue

        # Rename value column to a generic name
        value_col = [col for col in df.columns if col != 'date'][0]
        df = df[['date', value_col]]
        df = df.rename(columns={value_col: 'value'})

        dfs.append(df)

    if dfs:
        # Merge all dataframes on 'date' and sum values
        merged_df = pd.concat(dfs, axis=0)
        merged_df = merged_df.groupby('date', as_index=False)['value'].sum()
        merged_df['UCID'] = ucid

        final_list.append(merged_df)

# Combine all UCID-level results
if final_list:
    result_df = pd.concat(final_list, ignore_index=True)
    result_df = result_df[['UCID', 'date', 'value']]  # reorder columns

    # Save to CSV
    output_path = r"C:\Users\Administrator\Desktop\TAO_data\tvl_by_ucid.csv"
    result_df.to_csv(output_path, index=False)
    print("✅ Final CSV saved at:", output_path)
else:
    print("⚠️ No valid data found to process.")


Skipping hopr.csv: incorrect format
✅ Final CSV saved at: C:\Users\Administrator\Desktop\TAO_data\tvl_by_ucid.csv


In [2]:
# Combine all UCID-level results
if final_list:
    result_df = pd.concat(final_list, ignore_index=True)
    
    # Ensure proper column order
    result_df = result_df[['UCID', 'date', 'value']]

    # Convert 'date' to datetime for sorting
    result_df['date'] = pd.to_datetime(result_df['date'], errors='coerce')

    # Sort by UCID and then date
    result_df = result_df.sort_values(by=['UCID', 'date']).reset_index(drop=True)

    # Check if each (UCID, date) pair is unique
    duplicated = result_df.duplicated(subset=['UCID', 'date'], keep=False)
    if duplicated.any():
        dupes = result_df[duplicated]
        print("⚠️ Warning: Found duplicate (UCID, date) entries:")
        print(dupes)
    else:
        print("✅ All (UCID, date) pairs are unique.")

    # Save the final sorted file
    output_path = r"C:\Users\Administrator\Desktop\TAO_data\tvl_by_ucid.csv"
    result_df.to_csv(output_path, index=False)
    print("✅ Sorted and validated file saved at:", output_path)
else:
    print("⚠️ No valid data found to process.")


✅ All (UCID, date) pairs are unique.
✅ Sorted and validated file saved at: C:\Users\Administrator\Desktop\TAO_data\tvl_by_ucid.csv
